# 🎯 HW - Data Cleaning and Fitting
## Predicting Coolant Pump Failure using Messy Data Logs

⚠️ The primary coolant pump is a critical point of failure. If it goes down, the core overheats. The manager has access to logs from the pump, but the logs are notoriously messy— sensors drop offline, inputs get duplicated, and formats vary.

🚀 Your mission is to build an AI regression model that can read this chaotic data and predict whether a pump is running Safe (0) or heading for a Failure (1). But before you can train your AI, you have to survive the Data Pipeline Gauntlet. The hard part is not only cleaning the table. The failure class is rare, so a model can look good by predicting "safe" almost all the time. 

In this notebook you will build the full workflow from data cleaning to model fitting on imbalanced dataset. 
- This notebook runs top-to-bottom on **Google Colab**; visualization code is hidden behind a title (double-click to peek).
- Cells marked **🎯🎯 TASK 🎯🎯** are for you to complete.
- Most **🎯🎯 TASK 🎯🎯** blocks contain assertions to check and validate your results. 


## Part 0 — Setup

This notebook is **self-contained**: when you run the first cell it pulls the rest of the
exercise files (the pump logs dataset under `data/`) directly from the course repository. Run the setup cells below in order.

> ✅ The course repo is **public**, so no GitHub token is needed — the cell clones it directly.
> If you're running this notebook from inside a local clone of the repo, the setup cell skips the
> clone and just uses the files already on disk.

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo...")
        !git clone -q "$url"
    else:
        print("Updating the exercise repo...")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull - using the existing copy)"

# Locate this homework folder (holds helpers/ + data/) and make the helpers importable.
_HW = os.path.join("workshop", "homework")
for _root in [os.path.join(REPO_NAME, _HW), ".", _HW,
              os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "data", "hw_pump.csv")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the homework folder (workshop/homework with helpers/ + data/). "
        "On Colab, re-run this cell to clone it; locally, run the notebook from inside a clone of the FDD-WE0-public repo.")
sys.path.insert(0, os.path.join(os.getcwd(), "helpers"))   # make the helpers importable
print("Working directory:", os.getcwd())


**0.2 — Install dependencies.** All of these are already on Colab; this just guarantees
the right versions (and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r requirements_hw_fitting.txt

**0.3 — Import the libraries** we'll use throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, RANSACRegressor
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import resample

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

BLUE, ORANGE, GREEN, RED = "#6f7bf0", "#dd8452", "#55a868", "#c44e52"
print("Ready.")

EXPECTED_OUTLIER_REPORT = pd.read_csv("data/outlier_report.csv")
EXPECTED_OUTLIER_REPORT_RANSAC = pd.read_csv("data/outlier_report_ransac.csv")

### The core idea

Run the cell below for a pipeline of what you're about to do:

In [ ]:
#@title 📊 Pipeline workflow (double-click to view the code) { display-mode: "form" }
def pump_workflow_diagram():
    steps = [("🔍", "Inspect", "raw CSV logs"), ("🧹", "Clean", "formatting & fill values"),
             ("📉", "Outliers", "sensor spikes"), ("🤖", "Model fit", "baseline model"),
             ("📊", "Evaluate", "metrics to use"), ("🛠️", "Strategy", "data & model level")]
    grad = ["#667eea", "#6f7bf0", "#55a868", "#3fa7a0", "#dd8452", "#c56fbe"]
    blocks = ""
    for (ic, t, d), g in zip(steps, grad):
        blocks += (f'<div class="pl-step"><div class="pl-ic" style="background:linear-gradient(135deg,{g},{g}cc)">{ic}</div>'
                   f'<div class="pl-t">{t}</div><div class="pl-d">{d}</div></div>'
                   '<div class="pl-ar">➜</div>')
    blocks = blocks.rsplit('<div class="pl-ar">➜</div>', 1)[0]
    n = len(steps)
    model_i, strat_i = 3, n - 1
    w = 600
    mx, sx = w * (model_i + 0.5) / n, w * (strat_i + 0.5) / n
    loop = f'M {sx - 2:.0f} 6 C {sx - 58:.0f} 28, {mx + 58:.0f} 28, {mx + 2:.0f} 6'
    display(HTML(f'''
<style>
.pl{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#f0f4ff);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #e0e8f5}}
.pl-h{{font-size:20px;font-weight:800;color:#24324b;margin:0 0 14px}}
.pl-row{{display:flex;align-items:flex-start;flex-wrap:wrap;gap:0}}
.pl-step{{flex:1 1 96px;min-width:96px;text-align:center;padding:0 2px}}
.pl-ic{{width:48px;height:48px;border-radius:50%;margin:0 auto 7px;display:flex;align-items:center;
       justify-content:center;font-size:21px;color:#fff;box-shadow:0 6px 14px rgba(111,123,240,.35)}}
.pl-t{{font-weight:700;font-size:12px;color:#24324b}}
.pl-d{{font-size:10px;color:#52627a;margin-top:2px}}
.pl-ar{{display:flex;align-items:center;font-size:18px;color:#8da0cb;flex:0 0 14px;height:48px}}
.pl-loop{{width:100%;height:32px;margin-top:2px}}
</style>
<div class="pl"><div class="pl-h">⚙️ Coolant Pump Failure Prediction Pipeline</div><div class="pl-row">{blocks}</div>
<svg class="pl-loop" viewBox="0 0 600 32" preserveAspectRatio="none">
  <defs><marker id="pl-arr" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
    <path d="M0,0 L7,3.5 L0,7 Z" fill="#8da0cb"/></marker></defs>
  <path d="{loop}" stroke="#8da0cb" stroke-width="1.5"
        fill="none" marker-end="url(#pl-arr)"/>
</svg></div>'''))

pump_workflow_diagram()


## 🔍 Part 1 - Inspect the data logs

Start by loading the raw logs. Let us inspect some sample rows.


In [ ]:
DATA_PATH = "data/hw_pump.csv"
TARGET = "pump_failure_status"

raw_df = pd.read_csv(DATA_PATH)
print("shape:", raw_df.shape)
raw_df.head()

Our target column is named **`pump_failure_status`**. Let's check the data distribution of Safe and Failure logs.

In [ ]:
class_counts = raw_df[TARGET].value_counts().sort_index()
class_rate = raw_df[TARGET].value_counts(normalize=True).sort_index()
summary = pd.DataFrame({"rows": class_counts, "share": class_rate})
summary.index = ["safe (0)", "failure (1)"]

fig, ax = plt.subplots(figsize=(5.5, 4.0))
ax.bar(["safe", "failure"], class_counts.values, color=[BLUE, RED], edgecolor="white")
ax.set_title("Pump failure labels are heavily imbalanced", fontweight="bold", pad=14)
ax.set_ylabel("rows")
ax.set_ylim(0, class_counts.max() * 1.12)
for i, v in enumerate(class_counts.values):
    ax.text(i, v, f"{v:,}\n({v / len(raw_df):.1%})", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


### How do we find the mess?

From our data samples we can already identify some pain points in the data logs. But our data logs contain **10,000** rows which cannot be manually inspected. Let's systematically check all possible issues.

- formatting inconsistencies
- missing values
- duplicate rows
- extreme outliers

Let's make each problem visible before fixing it.


In [ ]:
# Infer the type of each value to get information 
def infer_value_type(value):
    if pd.isna(value):
        return "null"
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.isdigit() or (stripped.startswith("-") and stripped[1:].isdigit()):
            return "int"
        elif stripped.isdecimal():
            return "float"
        else:
            return "str"
    return type(value).__name__


type_report = pd.DataFrame({
    "column": raw_df.columns,
    "observed_value_type": [
        sorted(raw_df[col].map(infer_value_type).unique().tolist())
        for col in raw_df.columns
    ],
})

display(type_report)

More than one value type indicates messy data with formatting inconsistencies or missing values (null). 

**`String`** is notorious for being difficult to handle in data logs. Let's take another closer look for definitive conclusion.

In [ ]:
# 🎯🎯 Print all unique values of string (str) columns by replacing placeholder None values. What do you notice? 🎯🎯
for col in [None, None]:
        print(f"{col}: {None}")
# ---------------------------------------------- #

Lets now inspect how many columns have missing data values. Since these are sensor readings this is nothing unusual.

In [ ]:
print("Missing values")
missing_report = (
    raw_df.isna().sum().loc[lambda s: s > 0].rename("missing_rows").to_frame()
)
display(missing_report)

Next, we inspect if the logs have duplicated data.

In [ ]:
# 🎯🎯 Check if there are any duplicate rows in the logs. 🎯🎯
duplicate_count = None
# ---------------------------------------------- #

print(f"Number of duplicated rows: {duplicate_count}")
if duplicate_count:
    display(raw_df[raw_df.duplicated(keep=False)].head(10))

### Outlier Patrol: spotting suspicious sensor readings

Outliers are the dataset's little troublemakers: values that sit far away from the rest of the crowd. Sometimes they are real warning signs, like a pump sensor malfunctioning. Other times they are just messy logging glitches and typos.

Here we assess numeric outliers with the **interquartile range (IQR)**, which looks at the middle 50% of each column. Because failure rows can behave very differently from safe rows, we run the check twice: once for `pump_failure_status = 0` and once for `pump_failure_status = 1`.

Quartile context for each numeric feature in a group:

`Q1` = 25th percentile (lower quartile)  
`Q2` = 50th percentile (median)  
`Q3` = 75th percentile (upper quartile)  
`IQR = Q3 - Q1`

For each group, we build wide "fences" around that group's normal zone using:

`lower fence = Q1 - 3 × IQR`  
`upper fence = Q3 + 3 × IQR`

Any value outside those fences gets flagged as an **extreme outlier**. We do not delete anything yet; first we make a report so we can inspect which columns are acting weird and decide whether those values are useful failure signals or data gremlins.

In [ ]:
numeric_cols = [c for c in raw_df.select_dtypes(include="number").columns if c != TARGET]
outlier_reports = []

for failure_value in [0, 1]:
    group_df = raw_df[raw_df[TARGET] == failure_value]

    # 🎯🎯 Calculate the fences for each numeric column by replacing placeholder None values.🎯🎯
    q1 = group_df[numeric_cols].quantile(0.25)
    q3 = None
    iqr = None
    lower_fence = None
    upper_fence = None
    outlier_mask = group_df[numeric_cols].lt(lower_fence) | group_df[numeric_cols].gt(upper_fence)
    # ---------------------------------------------- #

    outlier_report = pd.DataFrame({
        "outlier_rows": outlier_mask.sum(),
        "outlier_rate_%": (outlier_mask.mean() * 100).round(2),
        "min": group_df[numeric_cols].min(),
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "max": group_df[numeric_cols].max(),
    })
    outlier_report = outlier_report.sort_values("outlier_rows", ascending=False, kind="stable")
    outlier_reports.append(outlier_report)

outlier_report = pd.concat(outlier_reports, ignore_index=True)

# Row order from sort_values() can tie-break differently across pandas versions/platforms,
# so compare the two reports order-independently (sort both by their values first).
_keys = ["outlier_rows", "outlier_rate_%", "min", "lower_fence", "upper_fence", "max"]
pd.testing.assert_frame_equal(
    outlier_report.sort_values(_keys).reset_index(drop=True),
    EXPECTED_OUTLIER_REPORT.sort_values(_keys).reset_index(drop=True),
    check_exact=False,
    atol=1e-12,
    rtol=1e-9,
)
print("Outlier report generated correctly.")

Run the below cell to display the outlier report you just created and observe which columns have outliers.

In [ ]:
print(f"Extreme numeric outliers when {TARGET} = 0")
display(outlier_report.iloc[:8])
print(f"Extreme numeric outliers when {TARGET} = 1")
display(outlier_report.iloc[8:16])

## 🧹 Part 2 - Clean the data logs

The goal is a table where each row has consistent types and physically usable values.

### Remove Duplicates and Split before applying Data Cleaning

Before cleaning, split the raw logs into train and validation sets. From this point onward, any cleaning choice that depends on observed values must be learned from the train split and then applied unchanged to validation.

In [ ]:
# 🎯🎯 Drop exact duplicates by replacing placeholder None value. NOTE: Remember to reset the index after dropping duplicates.🎯🎯
raw_df = None
# ---------------------------------------------- #
assert raw_df.duplicated().sum() == 0, "Duplicates found, try again"


raw_train_df, raw_val_df = train_test_split(
    raw_df,
    test_size=0.25,
    random_state=7,
    stratify=raw_df[TARGET],
)

raw_splits = {
    "train": raw_train_df.reset_index(drop=True),
    "val": raw_val_df.reset_index(drop=True),
}

split_summary = pd.DataFrame({
    name: {
        "rows": len(frame),
        "failure_rate": frame[TARGET].mean(),
    }
    for name, frame in raw_splits.items()
}).T

print(f"Rows left after dropping duplicates: {len(raw_df)}")
print("Raw split sizes before any cleaning:")
display(split_summary.round({"failure_rate": 4}))

In [ ]:
NUMERIC_FEATURES = [
    "inspection_rating", "runtime_hours", "flow_rate", "inlet_pressure", "outlet_pressure",
    "vibration_amplitude", "bearing_temperature", "motor_current", "motor_voltage",
]
CATEGORICAL_FEATURES = ["Pump_ID"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

SPLIT_NAMES = ["train", "val"]
split_logs = {name: df.copy() for name, df in raw_splits.items()}

### Consistency

Lets start with reformatting `Pump_ID` and `inspection_rating`.

In [ ]:
def reformat_pump_id(value):
    # Convert variants such as pmp-001, PMP001, and PMP_001 into PMP-001.
    digits = "".join(ch for ch in str(value).upper() if ch.isdigit())
    if not digits:
        return np.nan
    return f"PMP-{int(digits):03d}"

# 🎯🎯 Reformat the `Pump_ID` column to be consistent. Replace placeholder None value.🎯🎯
for data_frame in split_logs.values():
    data_frame["Pump_ID"] = None
# ---------------------------------------------- #

train_pump_ids = set(split_logs["train"]["Pump_ID"].unique())
val_pump_ids = set(split_logs["val"]["Pump_ID"].unique())
assert train_pump_ids == {"PMP-001", "PMP-002", "PMP-003", "PMP-004"}, "Wrong Pump_ID values, try again"
assert val_pump_ids == {"PMP-001", "PMP-002", "PMP-003", "PMP-004"}, "Wrong Pump_ID values, try again"
print("Reformatted Pump_ID values:", train_pump_ids)

In [ ]:
rating_words = {"three": 3, "four": 4, "five": 5}

# 🎯🎯 Refactor the `inspection_rating` column to be numeric. Replace placeholder None value.🎯🎯
for frame in split_logs.values():
    frame["inspection_rating"] = None
    frame["inspection_rating"] = pd.to_numeric(frame["inspection_rating"], errors="coerce")
# ---------------------------------------------- #

train_ratings = set(split_logs["train"]["inspection_rating"].unique())
val_ratings = set(split_logs["val"]["inspection_rating"].unique())
assert train_ratings.issubset({1, 2, 3, 4, 5}), "Wrong inspection rating values, try again"
assert val_ratings.issubset({1, 2, 3, 4, 5}), "Wrong inspection rating values, try again"
print("Reformatted inspection rating values:", train_ratings)

### Missing Values

For the missing sensor values, use the statistics table below to compare possible fill values. Then choose the statistic you think will work best. To avoid data leakage we will only calculate statistics on the training set and then apply it to the *unseen* validation set.

*Remember we have outliers which affects statistics!*

In [ ]:
train_logs = split_logs["train"]
val_logs = split_logs["val"]
MISSING_SENSOR_COLS = ["vibration_amplitude", "bearing_temperature", "motor_current", "motor_voltage"]

sensor_fill_options = pd.DataFrame({
    "mean": train_logs[MISSING_SENSOR_COLS].mean(),
    "median": train_logs[MISSING_SENSOR_COLS].median(),
    "mode": train_logs[MISSING_SENSOR_COLS].mode(dropna=True).iloc[0],
    "q1": train_logs[MISSING_SENSOR_COLS].quantile(0.25),
    "q3": train_logs[MISSING_SENSOR_COLS].quantile(0.75),
}).round(3)

print("Candidate replacement values for missing sensor readings, learned from train:")
display(sensor_fill_options)

In [ ]:
missing_before = pd.DataFrame({
    name: split_logs[name][MISSING_SENSOR_COLS].isna().sum()
    for name in SPLIT_NAMES
}).T

# 🎯🎯 Fill missing sensor values with the train-derived candidate replacement you deem most appropriate. Replace all None. 🎯🎯
sensor_fill_choice = None
sensor_fill_values = sensor_fill_options[sensor_fill_choice]
for frame in split_logs.values():
    frame[MISSING_SENSOR_COLS] = None
# ---------------------------------------------- #

missing_after = pd.DataFrame({
    name: split_logs[name][MISSING_SENSOR_COLS].isna().sum()
    for name in SPLIT_NAMES
}).T

print(f"Sensor missing values filled with train {sensor_fill_choice}")
print("Fill values used:\n",sensor_fill_values.to_string())
print("Missing before:")
display(missing_before)
print("Missing after:")
display(missing_after)

## 📉 Part 3- Discard sensor outliers with RANSAC

Most sensor readings move together along a steady trend; a few rows are spikes that do not fit that pattern. These spikes bias the dataset and skew the model learning process. 

**RANSAC (RANdom SAmple Consensus)** is a robust outlier detection approach and a very common step in data cleaning pipeline. It repeatedly fits a line to a tiny random subset of points, counts how many other points lie close to that line (**inliers**), and keeps the model supported by the largest inlier group. Outliers that never agree with the winning line are left out.

Try the interactive plot below: step through single trials or run RANSAC to completion to see mini-samples, inliers, and the final fit.

In [ ]:
#@title 📉 How RANSAC works (interactive — double-click to view code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid

_uid = "ransac-viz-" + uuid.uuid4().hex[:8]

display(HTML(f'''
<style>
#{_uid} {{
  font-family: system-ui, Segoe UI, Roboto, sans-serif;
  background: linear-gradient(135deg, #f6f8ff, #f0f4ff);
  border-radius: 18px;
  padding: 18px 20px 16px;
  margin: 8px 0;
  border: 1px solid #e0e8f5;
  color: #24324b;
}}
#{_uid} .rv-h {{
  font-size: 18px;
  font-weight: 800;
  margin: 0 0 4px;
}}
#{_uid} .rv-sub {{
  font-size: 12px;
  color: #52627a;
  margin: 0 0 12px;
  line-height: 1.45;
}}
#{_uid} .rv-body {{
  display: flex;
  flex-wrap: wrap;
  gap: 14px;
}}
#{_uid} .rv-panel {{
  flex: 1 1 220px;
  min-width: 200px;
}}
#{_uid} .rv-step {{
  background: #fff;
  border: 1px solid #e6ecf7;
  border-radius: 12px;
  padding: 10px 12px;
  margin-bottom: 8px;
  font-size: 12px;
  line-height: 1.45;
}}
#{_uid} .rv-step b {{ color: #3fa7a0; }}
#{_uid} .rv-msg {{
  background: #fff;
  border-left: 4px solid #667eea;
  border-radius: 8px;
  padding: 10px 12px;
  font-size: 12px;
  min-height: 52px;
  line-height: 1.5;
}}
#{_uid} .rv-plot-wrap {{
  flex: 2 1 360px;
  min-width: 280px;
  background: #fff;
  border: 1px solid #e6ecf7;
  border-radius: 14px;
  padding: 8px;
}}
#{_uid} svg {{
  width: 100%;
  height: auto;
  display: block;
}}
#{_uid} .rv-ctrl {{
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
  margin-top: 10px;
  align-items: center;
}}
#{_uid} button {{
  font-family: inherit;
  font-size: 12px;
  font-weight: 600;
  border: none;
  border-radius: 999px;
  padding: 7px 14px;
  cursor: pointer;
  color: #fff;
  background: linear-gradient(135deg, #667eea, #6f7bf0);
  box-shadow: 0 4px 10px rgba(102, 126, 234, .28);
}}
#{_uid} button.secondary {{
  background: linear-gradient(135deg, #55a868, #3fa7a0);
}}
#{_uid} button.ghost {{
  background: #eef2fb;
  color: #42526e;
  box-shadow: none;
  border: 1px solid #d8e0f0;
}}
#{_uid} button:disabled {{ opacity: .45; cursor: default; }}
#{_uid} label {{
  font-size: 11px;
  color: #52627a;
  display: flex;
  align-items: center;
  gap: 6px;
}}
#{_uid} input[type=range] {{ width: 110px; accent-color: #667eea; }}
#{_uid} .rv-legend {{
  display: flex;
  flex-wrap: wrap;
  gap: 12px;
  font-size: 11px;
  color: #52627a;
  margin-top: 6px;
}}
#{_uid} .rv-dot {{
  display: inline-block;
  width: 10px;
  height: 10px;
  border-radius: 50%;
  margin-right: 4px;
  vertical-align: -1px;
}}
#{_uid} .rv-stats {{
  font-size: 11px;
  color: #52627a;
  margin-top: 6px;
}}
</style>
<div id="{_uid}">
  <div class="rv-h">RANSAC — robust line fitting</div>
  <div class="rv-sub">
    Most sensor readings move together; spikes break ordinary least squares.
    RANSAC repeatedly fits lines to tiny random samples and keeps the model supported by the most inliers.
  </div>
  <div class="rv-body">
    <div class="rv-panel">
      <div class="rv-step"><b>1.</b> Start with noisy (x, y) pairs — most follow a trend, a few are outliers.</div>
      <div class="rv-step"><b>2.</b> Draw a <em>minimal sample</em> (here: 2 points) and fit a candidate line.</div>
      <div class="rv-step"><b>3.</b> Count <em>inliers</em>: points within a distance threshold of that line.</div>
      <div class="rv-step"><b>4.</b> Repeat many trials; the line with the most inliers wins.</div>
      <div class="rv-msg" id="{_uid}-msg">Click <b>New data</b>, then step through trials or run RANSAC to completion.</div>
      <div class="rv-stats" id="{_uid}-stats"></div>
    </div>
    <div class="rv-plot-wrap">
      <svg id="{_uid}-svg" viewBox="0 0 420 280" aria-label="RANSAC scatter plot"></svg>
      <div class="rv-legend">
        <span><span class="rv-dot" style="background:#667eea"></span>Inlier</span>
        <span><span class="rv-dot" style="background:#dd8452"></span>Outlier / spike</span>
        <span><span class="rv-dot" style="background:#c56fbe"></span>Mini-sample</span>
        <span><span class="rv-dot" style="background:#3fa7a0"></span>RANSAC line</span>
        <span><span class="rv-dot" style="background:#8da0cb;border:1px dashed #52627a"></span>OLS (not robust)</span>
      </div>
      <div class="rv-ctrl">
        <button type="button" class="ghost" id="{_uid}-new">New data</button>
        <button type="button" id="{_uid}-step">One RANSAC trial</button>
        <button type="button" class="secondary" id="{_uid}-run">Run to best fit</button>
        <button type="button" class="ghost" id="{_uid}-reset">Reset view</button>
        <label>Threshold
          <input type="range" id="{_uid}-thresh" min="4" max="22" value="12" step="1">
          <span id="{_uid}-thresh-val">12</span> px
        </label>
      </div>
    </div>
  </div>
</div>
<script>
(function() {{
  var root = document.getElementById("{_uid}");
  var svg = document.getElementById("{_uid}-svg");
  var msg = document.getElementById("{_uid}-msg");
  var stats = document.getElementById("{_uid}-stats");
  var threshInput = document.getElementById("{_uid}-thresh");
  var threshVal = document.getElementById("{_uid}-thresh-val");

  var W = 420, H = 280, pad = 36;
  var pts = [], trial = 0, maxTrials = 40;
  var best = null, current = null, sampleIdx = [];
  var rng = 0;

  function rand() {{
    rng = (rng * 1664525 + 1013904223) >>> 0;
    return rng / 4294967296;
  }}

  function trueLine(x) {{ return 0.55 * x + 18; }}

  function genData(seed) {{
    rng = seed >>> 0;
    pts = [];
    for (var i = 0; i < 28; i++) {{
      var x = 20 + rand() * 300;
      var y = trueLine(x) + (rand() - 0.5) * 14;
      pts.push({{ x: x, y: y, outlier: false }});
    }}
    for (var j = 0; j < 6; j++) {{
      pts.push({{
        x: 20 + rand() * 300,
        y: 20 + rand() * 220,
        outlier: true
      }});
    }}
    trial = 0;
    best = null;
    current = null;
    sampleIdx = [];
    setMsg("Fresh scatter generated. Try <b>One RANSAC trial</b> or <b>Run to best fit</b>.");
    draw();
  }}

  function toSvg(p) {{
    var xmin = 10, xmax = 330, ymin = 10, ymax = 250;
    return {{
      sx: pad + (p.x - xmin) / (xmax - xmin) * (W - 2 * pad),
      sy: H - pad - (p.y - ymin) / (ymax - ymin) * (H - 2 * pad)
    }};
  }}

  function fromModel(m) {{
    var xmin = 10, xmax = 330, ymin = 10, ymax = 250;
    var x1 = xmin, y1 = m.m * x1 + m.b;
    var x2 = xmax, y2 = m.m * x2 + m.b;
    var a = toSvg({{ x: x1, y: y1 }});
    var b = toSvg({{ x: x2, y: y2 }});
    return {{ x1: a.sx, y1: a.sy, x2: b.sx, y2: b.sy }};
  }}

  function fitTwo(i, j) {{
    var p1 = pts[i], p2 = pts[j];
    if (Math.abs(p2.x - p1.x) < 1e-6) return null;
    var m = (p2.y - p1.y) / (p2.x - p1.x);
    var b = p1.y - m * p1.x;
    return {{ m: m, b: b }};
  }}

  function distPx(p, model) {{
    var yHat = model.m * p.x + model.b;
    var a = toSvg(p);
    var b = toSvg({{ x: p.x, y: yHat }});
    return Math.hypot(a.sx - b.sx, a.sy - b.sy);
  }}

  function countInliers(model) {{
    var th = +threshInput.value;
    var n = 0, mask = [];
    for (var k = 0; k < pts.length; k++) {{
      var ok = distPx(pts[k], model) <= th;
      mask.push(ok);
      if (ok) n++;
    }}
    return {{ n: n, mask: mask }};
  }}

  function ols() {{
    var n = pts.length, sx = 0, sy = 0, sxx = 0, sxy = 0;
    for (var i = 0; i < n; i++) {{
      sx += pts[i].x; sy += pts[i].y;
      sxx += pts[i].x * pts[i].x;
      sxy += pts[i].x * pts[i].y;
    }}
    var den = n * sxx - sx * sx;
    if (Math.abs(den) < 1e-9) return null;
    var m = (n * sxy - sx * sy) / den;
    var b = (sy - m * sx) / n;
    return {{ m: m, b: b }};
  }}

  function pickSample() {{
    var i = Math.floor(rand() * pts.length);
    var j = Math.floor(rand() * pts.length);
    while (j === i) j = Math.floor(rand() * pts.length);
    return [i, j];
  }}

  function oneTrial() {{
    var pair = pickSample();
    var model = fitTwo(pair[0], pair[1]);
    if (!model) return oneTrial();
    var info = countInliers(model);
    sampleIdx = pair;
    current = {{ model: model, inliers: info.n, mask: info.mask }};
    trial++;
    if (!best || info.n > best.inliers) best = {{ model: model, inliers: info.n, mask: info.mask, sample: pair.slice() }};
    setMsg("Trial <b>" + trial + "</b>: mini-sample points highlighted in purple. "
      + "This candidate has <b>" + info.n + "</b> inliers within the threshold.");
    draw();
  }}

  function runAll() {{
    for (var t = trial; t < maxTrials; t++) oneTrial();
    current = best;
    sampleIdx = best ? best.sample : [];
    setMsg("Finished <b>" + maxTrials + "</b> trials. Best model: <b>" + best.inliers
      + "</b> inliers. Outliers (orange) are ignored for the green RANSAC line.");
    draw();
  }}

  function setMsg(html) {{ msg.innerHTML = html; }}

  function draw() {{
    var olsModel = ols();
    var parts = [
      '<rect width="' + W + '" height="' + H + '" fill="#fbfcff" rx="10"/>',
      '<line x1="' + pad + '" y1="' + (H-pad) + '" x2="' + (W-pad) + '" y2="' + (H-pad) + '" stroke="#d0d9ea"/>',
      '<line x1="' + pad + '" y1="' + pad + '" x2="' + pad + '" y2="' + (H-pad) + '" stroke="#d0d9ea"/>',
      '<text x="' + (W/2) + '" y="' + (H-8) + '" text-anchor="middle" font-size="11" fill="#52627a">sensor A</text>',
      '<text x="12" y="' + (H/2) + '" transform="rotate(-90 12 ' + (H/2) + ')" text-anchor="middle" font-size="11" fill="#52627a">sensor B</text>'
    ];

    if (olsModel) {{
      var ol = fromModel(olsModel);
      parts.push('<line x1="' + ol.x1 + '" y1="' + ol.y1 + '" x2="' + ol.x2 + '" y2="' + ol.y2
        + '" stroke="#8da0cb" stroke-width="2" stroke-dasharray="6 5" opacity="0.85"/>');
    }}

    var activeMask = current ? current.mask : null;
    for (var i = 0; i < pts.length; i++) {{
      var p = pts[i];
      var c = toSvg(p);
      var r = 5.5;
      var fill = p.outlier ? "#dd8452" : "#667eea";
      if (activeMask && !activeMask[i]) fill = "#f0c4a8";
      if (sampleIdx.indexOf(i) >= 0) {{ fill = "#c56fbe"; r = 7; }}
      parts.push('<circle cx="' + c.sx + '" cy="' + c.sy + '" r="' + r + '" fill="' + fill
        + '" stroke="#fff" stroke-width="1.2" opacity="0.95"/>');
    }}

    if (current) {{
      var ln = fromModel(current.model);
      parts.push('<line x1="' + ln.x1 + '" y1="' + ln.y1 + '" x2="' + ln.x2 + '" y2="' + ln.y2
        + '" stroke="#3fa7a0" stroke-width="3" stroke-linecap="round"/>');
    }}

    svg.innerHTML = parts.join("");
    var th = +threshInput.value;
    stats.textContent = "Trials run: " + trial + " / " + maxTrials
      + (best ? " · Best inlier count: " + best.inliers : "")
      + " · Distance threshold: " + th + " px";
  }}

  root.querySelector("#{_uid}-new").addEventListener("click", function() {{
    genData((Date.now() & 0xffff) + Math.floor(rand() * 1000));
  }});
  root.querySelector("#{_uid}-step").addEventListener("click", function() {{
    if (!pts.length) genData(42);
    if (trial >= maxTrials) {{ trial = 0; best = null; }}
    oneTrial();
  }});
  root.querySelector("#{_uid}-run").addEventListener("click", function() {{
    if (!pts.length) genData(42);
    trial = 0; best = null;
    runAll();
  }});
  root.querySelector("#{_uid}-reset").addEventListener("click", function() {{
    trial = 0; best = null; current = null; sampleIdx = [];
    setMsg("View reset. Data unchanged — run more trials or generate <b>New data</b>.");
    draw();
  }});
  threshInput.addEventListener("input", function() {{
    threshVal.textContent = this.value;
    if (current) {{
      var info = countInliers(current.model);
      current.inliers = info.n;
      current.mask = info.mask;
      if (best && best.model === current.model) {{
        best.inliers = info.n;
        best.mask = info.mask;
      }}
    }}
    draw();
  }});

  genData(2026);
}})();
</script>
'''))

We use **RANSAC** as a more dependable outlier removal method in the pump logs than our previous statistical verification approach during inspection. For each noisy sensor, we predict it from the other numeric readings with `RANSACRegressor` (see the [scikit-learn docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RANSACRegressor.html) if you want the full API). Rows with very large residuals—far from the robust line—are flagged and removed.

As this is a labeled log, we run the residual screen separately inside the safe and failure classes. That prevents the cleaner from treating the entire failure class as "bad data" simply because it is rare.

In [ ]:
def fit_ransac_outlier_rules(train_df, target_col=TARGET, residual_scale=8.0):
    keep = np.ones(len(train_df), dtype=bool)
    rules = []
    details = []

    for label in [0]:  # sorted(train_df[target_col].unique()):
        idx = train_df.index[train_df[target_col] == label]
        subset = train_df.loc[idx]
        class_keep = np.ones(len(subset), dtype=bool)

        for sensor in MISSING_SENSOR_COLS:
            predictors = [c for c in NUMERIC_FEATURES if c != sensor]
            model = RANSACRegressor(
                estimator=LinearRegression(),
                min_samples=0.65,
                random_state=0,
            )
            model.fit(subset[predictors], subset[sensor])
            residual = np.abs(subset[sensor] - model.predict(subset[predictors]))
            med = np.median(residual)
            mad = np.median(np.abs(residual - med))
            robust_sigma = 1.4826 * (mad if mad > 0 else np.std(residual) if np.std(residual) > 0 else 1.0)
            threshold = med + residual_scale * robust_sigma
            flagged = residual > threshold

            class_keep &= ~flagged.to_numpy()
            rules.append({
                "class": label,
                "sensor": sensor,
                "predictors": predictors,
                "model": model,
                "threshold": float(threshold),
            })
            details.append({
                "split": "train",
                "class": label,
                "sensor": sensor,
                "flagged_rows": int(flagged.sum()),
                "threshold": round(float(threshold), 3),
            })

        keep[idx] = class_keep

    return rules, keep, pd.DataFrame(details)


def apply_ransac_outlier_rules(df, rules, split_name, target_col=TARGET):
    keep = np.ones(len(df), dtype=bool)
    details = []

    for rule in rules:
        idx = df.index[df[target_col] == rule["class"]]
        subset = df.loc[idx]
        if subset.empty:
            flagged = pd.Series(False, index=idx)
        else:
            residual = np.abs(subset[rule["sensor"]] - rule["model"].predict(subset[rule["predictors"]]))
            flagged = residual > rule["threshold"]
            keep[idx] &= ~flagged.to_numpy()

        details.append({
            "split": split_name,
            "class": rule["class"],
            "sensor": rule["sensor"],
            "flagged_rows": int(flagged.sum()),
            "threshold": round(float(rule["threshold"]), 3),
        })

    return keep, pd.DataFrame(details)

In [ ]:
keep_masks = {"train": None, "val": None}
outlier_reports = {"train": None, "val": None}

# 🎯🎯 Fit RANSAC rules to the training set using the `fit_ransac_outlier_rules` function. 🎯🎯
ransac_rules, keep_masks["train"], outlier_reports["train"] = None
# ---------------------------------------------- #

# 🎯🎯 Apply RANSAC rules to the validation set using the `apply_ransac_outlier_rules` function. 🎯🎯
keep_masks["val"], outlier_reports["val"] = None
# ---------------------------------------------- #

outlier_report = pd.concat((outlier_reports["train"], outlier_reports["val"]), ignore_index=True)
pd.testing.assert_frame_equal(
    outlier_report,
    EXPECTED_OUTLIER_REPORT_RANSAC,
    check_exact=False,
    atol=1e-12,
    rtol=1e-9,
)

In [ ]:
# 🎯🎯 Remove extreme outliers from `split_logs` by using train-derived RANSAC rules. 🎯🎯
clean_splits = {"train": pd.DataFrame(), "val": pd.DataFrame()}
for name in SPLIT_NAMES:
    clean_splits[name] = None
# ---------------------------------------------- #

assert len(clean_splits["train"]) == 7230, "Train outliers not removed correctly, try again"
assert len(clean_splits["val"]) == 2400, "Validation outliers not removed correctly, try again"
clean_train_df = clean_splits["train"]
clean_val_df = clean_splits["val"]

In [ ]:
for name, frame in clean_splits.items():
    assert frame.duplicated().sum() == 0
    assert frame[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0
    assert set(frame["Pump_ID"].unique()) == {"PMP-001", "PMP-002", "PMP-003", "PMP-004"}
    assert set(frame["inspection_rating"].unique()).issubset({1, 2, 3, 4, 5})
print("Cleaning checks passed for train and val.")

print("\nFinal class balance by split:")
display(pd.DataFrame({
    name: frame[TARGET].value_counts().sort_index()
    for name, frame in clean_splits.items()
}).T.fillna(0).astype(int))


## 🤖 Part 4 - Baseline Model on Limited Features

Congratulations on cleaning the data logs and turning it into a valuable dataset! We will soon train a logistic regression on this cleaned dataset. But before that, let's consider the scenario where we did not go through the data cleaning pipeline and simply used available extractable data 🤔. 

`SELECTED_FEATURES` here considers as inputs only untouched columns during data cleaning process. Let's see how it performs.


In [ ]:
# Features we could have used from the raw data logs directly
SELECTED_FEATURES = ["runtime_hours",
                    "flow_rate",
                    "inlet_pressure",
                    "outlet_pressure"
                    ]
FEATURE_SET_NAME = "selected features"

missing_features = sorted(set(SELECTED_FEATURES) - set(ALL_FEATURES))
assert SELECTED_FEATURES, "Choose at least one feature."
assert not missing_features, f"Unknown feature(s): {missing_features}"

SELECTED_NUMERIC_FEATURES = [f for f in SELECTED_FEATURES if f in NUMERIC_FEATURES]
SELECTED_CATEGORICAL_FEATURES = [f for f in SELECTED_FEATURES if f in CATEGORICAL_FEATURES]

X_train = clean_train_df[SELECTED_FEATURES]
y_train = clean_train_df[TARGET].astype(int)
X_val = clean_val_df[SELECTED_FEATURES]
y_val = clean_val_df[TARGET].astype(int)

print("train rows:", len(X_train), "failure rate:", round(y_train.mean(), 4))
print("val rows:  ", len(X_val), "failure rate:", round(y_val.mean(), 4))

In [ ]:
# Dataloader pipeline
def split_feature_types(features):
    numeric_features = [f for f in features if f in NUMERIC_FEATURES]
    categorical_features = [f for f in features if f in CATEGORICAL_FEATURES]
    return numeric_features, categorical_features

def make_preprocessor(features):
    numeric_features, categorical_features = split_feature_types(features)
    transformers = []

    if numeric_features:
        transformers.append(("num", StandardScaler(), numeric_features))
    if categorical_features:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features))
    if not transformers:
        raise ValueError("Choose at least one numeric or categorical feature.")

    return ColumnTransformer(transformers)

# Logistic Regression Model
def make_logistic_pipeline(features, class_weight=None, **model_kwargs):
    return Pipeline([
        ("preprocess", make_preprocessor(features)),
        ("model", LogisticRegression(max_iter=100, random_state=0, class_weight=class_weight, **model_kwargs)),
    ])

# Evaluation Metrics
def calculate_metrics(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "predicted_failures": int(np.sum(y_pred)),
    }

In [ ]:
#🎯🎯 Complete the function that executes the model fitting pipeline and returns the model, predictions, and metrics 🎯🎯
def run_pipeline(name, model, X_train, y_train, X_val, y_val):
    model.fit(None, None)
    pred = None
    metrics = None
    return model, pred, metrics
# ---------------------------------------------- #

Let's run the above created pipeline on our selected features.


In [ ]:
baseline_model = make_logistic_pipeline(SELECTED_FEATURES)
baseline_model, baseline_pred, baseline_metrics = run_pipeline(
    f"baseline logistic - {FEATURE_SET_NAME}",
    baseline_model,
    X_train,
    y_train,
    X_val,
    y_val,
)
print(pd.Series(baseline_metrics).to_string())

What do you infer from the metrics? If accuracy is high that means we already have a well-working model!? So we just did all the data cleaning for nothing? 😲 Let's further visualize the predictions using a confusion matrix to be sure. 

A **confusion matrix** is a table that displays four key outcomes for any given class:
- True Positive (TP): The model correctly predicted a positive outcome.
- True Negative (TN): The model correctly predicted a negative outcome.
- False Positive (FP): The model incorrectly predicted a positive outcome (e.g., a false alarm).
- False Negative (FN): The model incorrectly predicted a negative outcome (e.g., a missed detection).

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay.from_predictions(
    y_val, baseline_pred, display_labels=["safe", "failure"], cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("Normal training: high accuracy, poor failure detection", fontweight="bold")
plt.tight_layout(); plt.show()

### What do you notice now?

For pump maintenance, the positive class (pump_failure_status = 1) is the expensive class. A missed failure can mean downtime, damage, or a safety incident. We need better metrics than just accuracy. That is why we track:

- When the model flags failure, how often is it right? Calculated as: $\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$

- How many real failures did it catch? Calculated as: $\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$

- Harmonic mean of precision and recall. Calculated as: $\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$


## 📊 Part 5 - Evaluation Metrics

Let's introduce new evaluation metrics **precision**, **recall**, and **F1-score** in our `metric_row` evaluation function. You can directly use existing [sklearn metrics functions](https://scikit-learn.org/stable/api/sklearn.metrics.html) like `precision_score`, `recall_score`, and `f1_failure`. 

In [ ]:
# 🎯🎯 Add precision, recall, and F1-score to the evaluation metrics by replacing placeholder None values. 🎯🎯
def calculate_metrics(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_failure": None,
        "recall_failure": None,
        "f1_failure": None,
        "predicted_failures": int(np.sum(y_pred)),
    }
# ---------------------------------------------- #

In [ ]:
baseline_model = make_logistic_pipeline(SELECTED_FEATURES)
baseline_model, baseline_pred, baseline_metrics = run_pipeline(
    f"baseline logistic - {FEATURE_SET_NAME}",
    baseline_model,
    X_train,
    y_train,
    X_val,
    y_val,
)
print(pd.Series(baseline_metrics).to_string())

Another handy evaluation metric tool is a **classification report**. It displays the main metrics— Accuracy, Precision, Recall, F1-Score, and Support— for each individual class in a structured table, providing a deeper look into a model's strengths and weaknesses.

In [ ]:
print(classification_report(y_val, baseline_pred, target_names=["safe", "failure"], digits=3, zero_division=0))

## 🛠️ Part 6 - Training Strategy 

Our dataset is heavily imbalanced: safe pumps vastly outnumber failures. A model can exploit that by predicting `pump_failure_status = 0` almost every time and still score high accuracy— exactly what the baseline did, while missing most real failures.

Class imbalance is common in fault detection. You can address it in two places: at the data level (change what the model sees during training) and at the model level (change how the model weighs each class). Below we try both.

### Data-level strategy: duplicate minority rows

A simple data-level fix is random oversampling: duplicate failure rows in the training set until the model sees about as many failure examples as safe examples.

This does not create new information. It changes the training pressure so the model is no longer rewarded for ignoring failures.

In [ ]:
def oversample_minority(X_train, y_train, target_col=TARGET, random_state=0):
    train = pd.concat(
        [X_train.reset_index(drop=True), y_train.reset_index(drop=True).rename(target_col)],
        axis=1,
    )
    majority = train[train[target_col] == 0]
    minority = train[train[target_col] == 1]

    minority_up = resample(
        minority,
        replace=True,
        n_samples=len(majority),
        random_state=random_state,
    )
    balanced = pd.concat([majority, minority_up]).sample(frac=1.0, random_state=random_state)
    return balanced[X_train.columns], balanced[target_col].astype(int)

X_train_over, y_train_over = oversample_minority(X_train, y_train)
print("Before oversampling:")
print(y_train.value_counts().sort_index())
print("\nAfter oversampling:")
print(y_train_over.value_counts().sort_index())

In [ ]:
# 🎯🎯 Run the pipeline with oversampled data by replacing placeholder None value. 🎯🎯
oversampled_model = make_logistic_pipeline(SELECTED_FEATURES)
oversampled_model, oversampled_pred, oversampled_metrics = None
# ---------------------------------------------- #
print(pd.Series(oversampled_metrics).to_string())
print(classification_report(y_val, oversampled_pred, target_names=["safe", "failure"], digits=3, zero_division=0))

### Model-level strategy: class-weighted learning

Instead of changing the rows, we can change the loss function. `class_weight="balanced"` tells logistic regression to penalize mistakes on rare failures more heavily.


In [ ]:
# 🎯🎯 Run the pipeline with class-weighted learning by setting class_weight="balanced". Replace placeholder None value.  🎯🎯
weighted_model = None
weighted_model, weighted_pred, weighted_metrics = None
# ---------------------------------------------- #
print(pd.Series(weighted_metrics).to_string())
print(classification_report(y_val, weighted_pred, target_names=["safe", "failure"], digits=3, zero_division=0))

In [ ]:
results = pd.DataFrame([baseline_metrics, oversampled_metrics, weighted_metrics])
metric_cols = ["accuracy", "precision_failure", "recall_failure", "f1_failure"]
results[metric_cols] = results[metric_cols].round(3)
results["model"] = ["baseline - selected features", "oversampled - selected features", "weighted - selected features"]
results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(results))
width = 0.22
for offset, metric, color in [
    (-width, "accuracy", BLUE),
    (0, "recall_failure", RED),
    (width, "f1_failure", GREEN),
]:
    ax.bar(x + offset, results[metric], width=width, label=metric, color=color, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(results["model"], rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Imbalance strategies trade accuracy for failure recall", fontweight="bold")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, pred, title in [
    (axes[0], baseline_pred, "baseline"),
    (axes[1], oversampled_pred, "oversampled"),
    (axes[2], weighted_pred, "class weighted"),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_val, pred, display_labels=["safe", "failure"], cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(title)
plt.tight_layout(); plt.show()

## ⚙️ Part 7 - Model Fitting on our Fully Cleaned Dataset

The imbalance strategies above fix the metric trap, but they do not replace feature quality. The actual sensor data logs contains strong failure signals like pressure, vibration, temperature, and current. Train the same normal model with all cleaned features and compare.


In [ ]:
X_train_full = clean_train_df[ALL_FEATURES]
X_val_full = clean_val_df[ALL_FEATURES]

# 🎯🎯 Run the pipeline with all cleaned features 🎯🎯
full_model = None
full_model, full_pred, full_metrics = None

# Optional Bonus: Explore the training strategies discussed above with the full dataset. 
# Use this code block as a free playing field.
# ---------------------------------------------- #

print(pd.Series(full_metrics).to_string())
print(classification_report(y_val, full_pred, target_names=["safe", "failure"], digits=3, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay.from_predictions(
    y_val, full_pred, display_labels=["safe", "failure"], cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("All features", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
final_results = pd.DataFrame([baseline_metrics, oversampled_metrics, weighted_metrics, full_metrics])
final_results[metric_cols] = final_results[metric_cols].round(3)
final_results["model"] = ["baseline - selected features", "oversampled - selected features", "weighted - selected features", "best - all features"]
final_results

## 📦 Wrap-up

You walked the full coolant-pump pipeline—from messy logs to a model that can actually flag failures:

1. **Inspect** the raw logs: class balance, dtypes, and suspicious sensor readings (IQR outlier patrol).
2. **Clean** the table before fitting: standardize pump IDs, fix mixed types, fill sensor dropouts, and remove duplicates—after a stratified train/validation split so validation stays honest.
3. **Screen outliers** with RANSAC: fit robust sensor relationships on the training set and drop rows with extreme residuals.
4. **Baseline fit** on a thin feature slice to see how much signal you lose when you skip cleaning and use only easy columns.
5. **Evaluate beyond accuracy** on this rare-event problem: precision, recall, and F1 for the failure class matter more than headline accuracy.
6. **Handle imbalance** with data-level oversampling and model-level balanced penalization.
7. **Refit on the full cleaned telemetry** and compare strategies side by side—the sensor features carry the real failure signal.

**Your final step:** run the cell below. It writes `solution_part1.py` into `workshop/solutions/` (and a copy in the repo root). Relaunch the control panel — it will load your `PumpSolution` via `get_solution()` and call `.clean(df)` and `.predict(df)` on live pump logs instead of the built-in reference answer.

In a real deployment, you would also tune the decision threshold from business costs: a false alarm costs an inspection, but a missed failure can cost downtime and safety risk.


## 📦 Package your solution for the control panel

You cleaned the logs step by step above. Now collect those exact steps into one function,
`clean_pump_log`, so the **Nuclear Central Manager** control panel can replay *your* cleaning on
live pump telemetry. The export cell below captures this function (plus your RANSAC screen and
model builders) into `solution_part1.py` — drop that into `workshop/solutions/` and relaunch the panel.

In [ ]:
def clean_pump_log(df, medians=None):
    """Package the cleaning you did above into one reusable function.

    Collect the SAME steps you applied to the splits: canonical Pump_IDs, numeric ratings,
    train-median sensor impute, de-duplicate.
    """
    rating_words = {"three": 3, "four": 4, "five": 5}
    out = df.copy()

    # 🎯🎯 Repeat your cleaning steps, in order. Replace each None. 🎯🎯
    out["Pump_ID"] = None
    out["inspection_rating"] = None
    out["inspection_rating"] = pd.to_numeric(out["inspection_rating"], errors="coerce")
    if medians is None:
        medians = out[MISSING_SENSOR_COLS].median()
    out[MISSING_SENSOR_COLS] = None
    # ---------------------------------------------- #

    return out.drop_duplicates().reset_index(drop=True)


# self-check: re-clean the raw log in one call — the same result your step-by-step cells produced.
_check = clean_pump_log(raw_df)
assert _check[MISSING_SENSOR_COLS].isna().sum().sum() == 0, "sensors should be imputed"
assert set(_check["Pump_ID"].unique()) == {"PMP-001", "PMP-002", "PMP-003", "PMP-004"}, "ids should be canonical"
print("clean_pump_log packages your cleaning ✅")

In [ ]:
# 📦 Export YOUR notebook as solution_part1.py for the Nuclear Central Manager control panel.
# Captures the SOURCE of the functions you wrote (infer_value_type, reformat_pump_id,
# clean_pump_log, fit_ransac_outlier_rules, apply_ransac_outlier_rules, and the model builders).
# A thin PumpSolution wrapper retrains them on the plant's historical log at load time
# (the notebook itself has no access to that generator).
import inspect, os

HEADER = """# Part 1 solution — coolant-pump cleaning + failure prediction.
# Exported from the HW1 notebook: the functions below are YOUR code.
import os, sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression, RANSACRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find the plant's data generator (lives in the workshop backend) from wherever this file sits.
for _rel in ("../backend", "../../workshop/backend", "workshop/backend", "."):
    _cand = os.path.abspath(os.path.join(os.path.dirname(__file__), _rel))
    if os.path.exists(os.path.join(_cand, "datagen", "pump_gen.py")):
        sys.path.insert(0, _cand)
        break
try:
    from datagen import pump_gen
except Exception:
    pump_gen = None

TARGET = pump_gen.TARGET if pump_gen is not None else "pump_failure_status"
NUMERIC_FEATURES = ["inspection_rating", "runtime_hours", "flow_rate", "inlet_pressure",
                    "outlet_pressure", "vibration_amplitude", "bearing_temperature",
                    "motor_current", "motor_voltage"]
CATEGORICAL_FEATURES = ["Pump_ID"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
MISSING_SENSOR_COLS = ["vibration_amplitude", "bearing_temperature", "motor_current", "motor_voltage"]
"""

WRAPPER = """

class PumpSolution:
    \"\"\"Trains YOUR cleaning + model on the plant's historical log, then scores live telemetry.\"\"\"

    def __init__(self):
        self._medians = None
        self._model = None
        self._fit_on_history()

    def clean(self, df):
        return clean_pump_log(df, self._medians)

    def predict(self, df):
        cleaned = self.clean(df)
        return self._model.predict(cleaned[ALL_FEATURES]).astype(int)

    def _fit_on_history(self):
        if pump_gen is None:
            raise ImportError(
                "pump_gen not found — place solution_part1.py in workshop/solutions/ "
                "(next to the workshop backend).")
        history = pump_gen.generate(seed=0)
        messy = pump_gen.make_messy(history.drop(columns=[\"failure_mode\"]), seed=11)
        cleaned = clean_pump_log(messy)
        self._medians = cleaned[MISSING_SENSOR_COLS].median()
        cleaned = clean_pump_log(messy, self._medians)
        _, keep, _ = fit_ransac_outlier_rules(cleaned)
        train_df = cleaned.loc[keep].reset_index(drop=True)
        self._model = make_logistic_pipeline(ALL_FEATURES, class_weight=\"balanced\")
        self._model.fit(train_df[ALL_FEATURES], train_df[TARGET].astype(int))


def get_solution():
    return PumpSolution()
"""

try:
    fns = (infer_value_type, reformat_pump_id, clean_pump_log,
           fit_ransac_outlier_rules, apply_ransac_outlier_rules,
           split_feature_types, make_preprocessor, make_logistic_pipeline)
    bodies = [inspect.getsource(fn) for fn in fns]
except NameError as exc:
    raise NameError(
        "Run the cells that define the cleaning + model functions before exporting."
    ) from exc

source = HEADER + "\n" + "\n\n".join(b.rstrip() + "\n" for b in bodies) + WRAPPER

targets = []
panel = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "solutions", "solution_part1.py"))
if os.path.isdir(os.path.dirname(panel)):
    targets.append(panel)
targets.append(os.path.abspath(os.path.join(os.getcwd(), "solution_part1.py")))

for dest in dict.fromkeys(targets):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "w") as f:
        f.write(source)
    print("Wrote", dest)

print("\nDrop solution_part1.py into workshop/solutions/ (done already if a path above is there),")
print("then relaunch the control panel — it scores live pump telemetry with YOUR cleaning + model.")

try:
    from google.colab import files
    files.download("solution_part1.py")
except Exception:
    pass